# The stratum server: one reuse cache for many runs

A stratum server is one long-lived Python process that runs pipelines for its
clients, which share its reuse cache. Later runs can reuse eligible reads,
features, and fitted models while their entries remain cached.

This notebook is the client: it builds skrub DAGs and sends them to the server,
which grid-searches or evaluates them. The server runs in a terminal.

## 1. Start the server in a terminal

From the repository root, with the `server` extra installed
(`pip install -e ".[server]"`). In PyCharm, its terminal already uses the project's
interpreter; the notebook's kernel must use the same one:

```bash
python -m stratum serve
```

It prints `stratum server listening on 127.0.0.1:<port> (reuse cache ..., 0 entries)`
and keeps running. It writes its address and a token under `~/.stratum/server/`,
where the client finds them. Cached results stay in memory until eviction,
a cache clear, or server shutdown. Use `--memory-fraction 0.1` to set the budget to
10% of host memory; entries are evicted in least-recently-used order.

## 2. Connect

In [ ]:
from pathlib import Path

from stratum import StratumClient

client = StratumClient()  # the local server, found under ~/.stratum/server
client.health()

## 3. The data

A synthetic sales table of a million rows, written next to this notebook. The DAG
names it by its absolute path, so the server reads the same file.

In [ ]:
import numpy as np
import pandas as pd

# examples/server/, whether the kernel starts there or in the project root.
HERE = next(d for d in (Path.cwd(), Path.cwd() / "examples" / "server")
            if (d / "server_client.ipynb").exists())
DATA = HERE / "sales.csv"


def make_sales(path, n=1_000_000):
    rng = np.random.default_rng(0)
    store = rng.integers(0, 200, n)
    price = rng.gamma(2.0, 10.0, n)
    quantity = rng.poisson(3, n) + 1
    pd.DataFrame({
        "store": store,
        "price": price,
        "quantity": quantity,
        "discount": rng.random(n) * 0.3,
        "revenue": price * quantity * (1 + store / 400) + rng.normal(0, 5, n),
    }).to_csv(path, index=False)


if not DATA.exists():
    make_sales(DATA)
pd.read_csv(DATA, nrows=5)

## 4. The pipeline

Built here, lazily: nothing runs in this kernel. It reads the sales table, derives
features and grid-searches the alpha of a Ridge model with 3-fold cross-validation.
The client pickles the DAG; library functions such as `pd.read_csv` and `np.log1p`
travel by name, and the server uses its own pandas and numpy.

This file source is keyed by its path, size, and modification time; changes to
that metadata change the key. Downstream keys include each operator's
implementation, parameters, and input keys. In-memory inputs are not keyed.

In [ ]:
import skrub
from sklearn.linear_model import Ridge

import stratum as st


def sales_table():
    sales = st.as_data_op(str(DATA)).skb.apply_func(pd.read_csv)
    return sales.assign(net_price=sales["price"] * (1 - sales["discount"]),
                        log_quantity=sales["quantity"].skb.apply_func(np.log1p))


def pipeline(alphas):
    with skrub.config_context(eager_data_ops=False):
        sales = sales_table()
        X = sales[["store", "price", "discount", "net_price", "log_quantity"]].skb.mark_as_X()
        y = sales["revenue"].skb.mark_as_y()
        return X.skb.apply(Ridge(alpha=skrub.choose_from(alphas, name="alpha")), y=y)

`search` sends the pipeline to the server and returns its results table. It also
reports what the run took from the cache (hits) and what it left in it (admitted),
from the server's statistics before and after.

In [ ]:
import time


def search(*alphas):
    before = client.stats()["stats"]
    start = time.perf_counter()
    results = client.grid_search(pipeline(list(alphas)), cv=3, scoring="r2")
    after = client.stats()["stats"]
    print(f"{time.perf_counter() - start:.2f}s: "
          f"{after['hits'] - before['hits']} values from the cache, "
          f"{after['puts'] - before['puts']} added to it")
    return results

## 5. A first search fills the cache

In [ ]:
search(1, 10_000)

## 6. Other alphas: the features come from the cache

The table and its features do not depend on alpha and can be reused if still
cached. The new alpha values require new model fits.

In [ ]:
search(100_000, 1_000_000)

## 7. The same alphas: the fitted models come from the cache too

Repeating the same alphas can reuse each fold's fitted Ridge and its output,
avoiding refitting while those entries remain cached.

In [ ]:
search(100_000, 1_000_000)

## 8. A query: evaluate

A DAG without X and y is a query: `evaluate` returns the table it computes. It reads
the same sales table as the searches and can reuse its cached read.

`share_of_total` is defined in this notebook, so the server has no module to import
it from: the client sends it by value, with the DAG (as it would a lambda, or a
helper module next to the notebook). The function remains opaque, so its output
and downstream operations have no cache key.

In [ ]:
def share_of_total(table):
    return table.assign(share=table["revenue"] / table["revenue"].sum())


with skrub.config_context(eager_data_ops=False):
    sales = sales_table()
    per_store = (sales.groupby("store", as_index=False)
                 .agg(revenue=("revenue", "sum"), orders=("quantity", "size"))
                 .sort_values("revenue", ascending=False)
                 .skb.apply_func(share_of_total))

before = client.stats()["stats"]["hits"]
top = client.evaluate(per_store)
print(client.stats()["stats"]["hits"] - before, "values from the cache")
top.head()

## 9. The cache

In [ ]:
{key: value for key, value in client.stats().items() if key != "stats"}

In [ ]:
client.stats()["stats"]

## 10. Clear the cache

`clear` removes every cached entry. Lifetime statistics remain available, so the
next run's work can still be measured by comparing counters before and after.
The terminal commands `python -m stratum stats` and `python -m stratum clear`
provide the same operations.


In [ ]:
client.clear()


In [ ]:
client.stats()


## 11. Stop the server

`shutdown` waits for queued jobs and stops the server. Its in-memory cache is lost;
starting the server again starts with an empty cache.

The server also runs whole scripts: `python -m stratum submit script.py [args]`,
or `client.submit(Path("script.py"), [...])`. Strings are Python source code.


In [ ]:
client.shutdown()


In [ ]:
client.close()
